In [1]:
import json
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.http.models import PointStruct, VectorParams, Distance
from qdrant_client.models import VectorParams, Distance
import uuid


In [2]:
# Load the json data 
data_path = "../../data/json_data/vision_data_metadata.json"
with open(data_path, "r", encoding="utf-8") as f:
    chunks = json.load(f)

In [3]:
chunks 

[{'title': 'Figure 3-1',
  'content': 'The image is a diagram labeled "Figure 3-1 Nutrient Absorption in the Gastrointestinal Tract." It illustrates the absorption of various nutrients throughout different sections of the gastrointestinal tract, including the esophagus, duodenum, stomach, jejunum, ileum, and colon. The diagram lists specific nutrients absorbed in each segment:\n\n- **Esophagus**: Calcium, Phosphorus, Magnesium, Iron, Copper, Selenium, Thiamin, Riboflavin, Niacin, Biotin, Folate, Vitamins A, D, E, K.\n- **Duodenum**:',
  'metadata': {'chapter': 'Surgical Nutrition',
   'section': 'OPTIONS FOR NUTRITIONAL THERAPY',
   'subsection': 'Digestive Tract Surgery',
   'is_figure': True,
   'is_table': False,
   'info_source': 'essentials-of-general-surgery-and-surgical-specialties Official Book'}},
 {'title': 'Figure 4-1',
  'content': 'The image is labeled "Figure 4-1 The Lethal Triad." It depicts a triangular diagram with three vertices labeled as follows:\n\n- **Acidosis** a

In [4]:
model = SentenceTransformer('all-MiniLM-L6-v2')  # or any other Sentence Transformer model

In [5]:

"""texts = [item["content"] for item in chunks]
embeddings = model.encode(texts, show_progress_bar=True)"""

'texts = [item["content"] for item in chunks]\nembeddings = model.encode(texts, show_progress_bar=True)'

In [6]:
client = QdrantClient(host="localhost", port=6333, timeout=60.0)

In [10]:

collection_name = "surgical_collection_multimodal"
if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name,
        vectors_config=VectorParams(size=384, distance=Distance.COSINE)
    )

In [ ]:
from tqdm import tqdm
import uuid
from qdrant_client.models import PointStruct

# Function to chunk list
def chunk_list(data, chunk_size):
    for i in range(0, len(data), chunk_size):
        yield data[i:i + chunk_size]

points = []

# Step 1: Encode and prepare points
for item in tqdm(chunks, desc="🔄 Embedding and preparing points"):
    embedding = model.encode(item["content"]).tolist()

    metadata = item["metadata"]
    payload = {
        "title": item["title"],
        "content": item["content"],
        "chapter": metadata.get("chapter"),
        "section": metadata.get("section"),
        "subsection": metadata.get("subsection"),
        "is_figure": metadata.get("is_figure", False),
        "is_table": metadata.get("is_table", False)
    }

    points.append(
        PointStruct(
            id=str(uuid.uuid4()),
            vector=embedding,
            payload=payload
        )
    )


🔄 Embedding and preparing points:   0%|          | 0/475 [00:00<?, ?it/s]

🔄 Embedding and preparing points: 100%|██████████| 475/475 [00:04<00:00, 96.94it/s] 


In [ ]:
# Step 2: Upload in batches (e.g., 300 points per batch depending on size)
batch_size = 300
print("🚀 Uploading to Qdrant in batches...")

for batch in tqdm(chunk_list(points, batch_size), total=len(points) // batch_size + 1, desc="📤 Uploading"):
    client.upsert(collection_name=collection_name, points=list(batch))

print("✅ All points uploaded successfully!")

🚀 Uploading to Qdrant in batches...


📤 Uploading: 100%|██████████| 2/2 [00:00<00:00,  4.58it/s]

✅ All points uploaded successfully!


In [ ]:
info = client.get_collection(collection_name="surgical_collection_multimodal")
print(info)


status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> vectors_count=None indexed_vectors_count=0 points_count=4187 segments_count=8 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=384, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=20000, flush_interval_sec=5, max_optimization_threads=None), wal_config=WalConfig(wal_capacity_mb=32, wal_segments_ahead=0), quantization_config=Non

In [13]:
count = client.count(collection_name="surgical_collection_multimodal", exact=True).count
print(f"🔍 Total points in collection: {count}")

🔍 Total points in collection: 4187


In [11]:
query = "preoperative evaluation and surgical risk"
query_vector = model.encode(query).tolist()

results = client.search(
    collection_name="surgical_collection",
    query_vector=query_vector,
    limit=5,
)

for result in results:
    print("Score:", result.score)
    print("Title:", result.payload.get("title"))
    print("Content:", result.payload.get("content")[:300], "...\n")


C:\Users\jihed\AppData\Local\Temp\ipykernel_12132\2987550945.py:4: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  results = client.search(


UnexpectedResponse: Unexpected Response: 404 (Not Found)
Raw response content:
b'{"status":{"error":"Not found: Collection `surgical_collection` doesn\'t exist!"},"time":2.3e-6}'

In [35]:
scroll_result = client.scroll(
    collection_name="surgical_collection",
    limit=5
)

for point in scroll_result[0]:
    print("Point ID:", point.id)
    print("Title:", point.payload.get("title"))
    print("Content preview:", point.payload.get("content")[:200], "\n")


Point ID: 002b2e31-1cd7-4730-b916-16e72b864705
Title: C.
Content preview: Pyogenic pericarditis  

Point ID: 003d5ed2-450f-48ce-8f69-9db984b44ac6
Title: EXTREMITY TRAUMA
Content preview: Extremity injuries are common in both blunt and penetrating trauma and may range in severity from trivial to limb and even life threatening. During the primary survey, potentially lethal injuries such 

Point ID: 004647cc-0f99-42c4-bba0-25870326d7b9
Title: Figure 20-1
Content preview: Arterial anatomy of the liver. Closeup of structures contained in the hepatoduodenal ligament. Tributaries of bile duct and branches of common hepatic artery. (Reprinted with permission from Sauerland 

Point ID: 00670cbc-097f-44fc-8500-68ace75dd5b3
Title: A.
Content preview: Percutaneous biopsy for pathologic evaluation and diagnosis  

Point ID: 00702479-c988-4eb0-93b1-46687031dbd1
Title: 1.
Content preview: Young children tend to have a uniform response to infection. Whether they have streptococcal pharyngitis, pneumon

## Retrieving step 

In [14]:
from qdrant_client.http.models import Filter, SearchRequest
from sentence_transformers import SentenceTransformer

def retrieve_relevant_chunks(query, model, client, collection_name, top_k=10):
    """
    Retrieve top-k relevant chunks from Qdrant given a user query.
    """
    # Step 1: Embed the user query
    query_vector = model.encode(query).tolist()

    # Step 2: Search in Qdrant
    search_result = client.search(
        collection_name=collection_name,
        query_vector=query_vector,
        limit=top_k
    )

    # Step 3: Parse and return results
    relevant_chunks = []
    for point in search_result:
        payload = point.payload
        relevant_chunks.append({
            "title": payload.get("title"),
            "content": payload.get("content"),
            "metadata": {
                "chapter": payload.get("chapter"),
                "section": payload.get("section"),
                "subsection": payload.get("subsection"),
                "is_figure": payload.get("is_figure", False),
                "is_table": payload.get("is_table", False)
            },
            "score": point.score
        })

    return relevant_chunks


In [15]:
query = """Which of the following statements is TRUE regarding the current approach to hepatic dysfunction in the preoperative setting?

A. Routine liver biochemical testing is recommended for all asymptomatic patients undergoing surgery.
B. Nonalcoholic fatty liver disease has become the least common cause of chronic liver disease in the U.S.
C. Liver biopsy is no longer used for staging liver disease.
D. Imaging and serologic tests are often sufficient for diagnosis and perioperative risk assessment."""

top_chunks = retrieve_relevant_chunks(query, model, client, collection_name)

for idx, chunk in enumerate(top_chunks):
    print(f"--- Chunk #{idx+1} ---")
    print("Title:", chunk["title"])
    print("Content:", chunk["content"][:300])  # print first 300 chars
    print("Score:", chunk["score"])
    print("Metadata:", chunk["metadata"])
    print()

--- Chunk #1 ---
Title: THE PATIENT WITH HEPATIC DYSFUNCTION
Content: Hepatic dysfunction was traditionally seen among patients with alcoholic hepatitis or chronic viral hepatitis. Although the incidence of these conditions has not changed, the overall number of patients with hepatic dysfunction has significantly increased with the obesity epidemic. Nonalcoholic fatty
Score: 0.7720748
Metadata: {'chapter': 'Patients', 'section': 'THE PATIENT WITH HEPATIC DYSFUNCTION', 'subsection': None, 'is_figure': False, 'is_table': False}

--- Chunk #2 ---
Title: 5.
Content: Patients with advanced liver disease are at an increased risk of mortality associated with surgery. All of the following are true regarding surgery in patients with advanced liver disease except: 
Score: 0.64934033
Metadata: {'chapter': 'Patients', 'section': 'SUGGESTED READINGS', 'subsection': None, 'is_figure': False, 'is_table': False}

--- Chunk #3 ---
Title: LIVER FAILURE
Content: Complications of chronic liver failure tha

C:\Users\jihed\AppData\Local\Temp\ipykernel_4556\890865426.py:12: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  search_result = client.search(


In [41]:
##LLM Integration

In [16]:
def build_prompt_with_context(query, retrieved_chunks, max_tokens=2000):
    """
    Builds a prompt for the LLM using the retrieved context chunks and the user's query.
    """
    context = ""
    current_tokens = 0

    # Add chunks one by one while respecting max_tokens
    for chunk in retrieved_chunks:
        chunk_text = f"### Title: {chunk['title']}\n{chunk['content']}\n\n"
        chunk_tokens = len(chunk_text.split())

        if current_tokens + chunk_tokens > max_tokens:
            break

        context += chunk_text
        current_tokens += chunk_tokens

    # Final prompt
    prompt = (
        "You are a helpful medical assistant trained on surgical knowledge.\n"
        "Answer the question based only on the following context:\n\n"
        f"{context}\n"
        "---\n"
        f"Question: {query}\n"
        "Answer:"
    )

    return prompt


In [17]:
query ="""what is the evaluation of  THE ORAL CAVITY AND PHARYNX """
retrieved = retrieve_relevant_chunks(query, model, client, collection_name)
prompt = build_prompt_with_context(query, retrieved)

print(prompt)

You are a helpful medical assistant trained on surgical knowledge.
Answer the question based only on the following context:

### Title: Physical Examination
Physical examination of the oral cavity and pharynx is facilitated by ample lighting and the use of tongue blades for retraction of the cheeks and tongue. Congenital, inflammatory, or neoplastic abnormalities should be sought. Complete inspection includes full examination of the buccal mucosal recesses, the palate, the floor of mouth, and the dental alveolar structures. The standard instruction, movement of the tongue and palate should be observed for symmetry. Most areas of the pharynx can be inspected with direct vision, mirrors, or fiberoptic endoscopes. Digital palpation of the oral cavity and oropharynx is mandatory because many lesions may be difficult or impossible to visualize but have palpable induration or irregularity. 

### Title: Figure 27-13
Sagittal view of the pharynx. (From Lawrence PF. Essentials of Surgical Speci

C:\Users\jihed\AppData\Local\Temp\ipykernel_4556\890865426.py:12: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  search_result = client.search(


In [20]:
import requests

def query_mistral(prompt, model="mistral"):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": model,
            "prompt": prompt,
            "stream": False
        }
    )
    return response.json()["response"]

In [21]:
answer = query_mistral(prompt)
print("💬 Answer:", answer)

💬 Answer:  Evaluation of the oral cavity and pharynx involves a comprehensive physical examination facilitated by adequate lighting and use of tongue blades for retraction. The examination includes inspection of various areas such as buccal mucosal recesses, the palate, the floor of mouth, and dental alveolar structures. The symmetry of movement of the tongue and palate should be observed. Direct vision, mirrors, or fiberoptic endoscopes can be used for inspection. Digital palpation is mandatory as many lesions may not be visible but have palpable induration or irregularity. For physical examination of the larynx and hypopharynx, special equipment like indirect laryngoscopy (requires a head light and a laryngeal mirror) or flexible fiberoptic laryngoscope are used. A sensitive gag reflex may require topical spray anesthetic for comfortable examination. In case of mandibular fracture, the evaluation focuses on assessing the airway, dental occlusion, abnormal position and mobility of tee